# 05 - Feature Engineering

## Objective
Turn raw attrition data into model-ready features:
- Drop columns that would leak the answer or carry no signal
- Encode categorical variables numerically
- Engineer new features with business/statistical rationale
- Produce a clean train-ready dataset

## Leakage columns removed
- **EmployeeNumber** — unique ID, not predictive
- **MonthlyRate** — redundant with MonthlyIncome (rate vs salary)
- **DailyRate** — same pay period issue as MonthlyRate
- **HourlyRate** — same pay period issue

## Engineered features
- **income_per_year** — MonthlyIncome * 12 / max(TotalWorkingYears, 1) → earning power relative to experience
- **years_since_promotion_ratio** — YearsSinceLastPromotion / max(TotalWorkingYears, 1) → career stagnation signal
- **satisfaction_score** — mean of JobSatisfaction + EnvironmentSatisfaction + WorkLifeBalance → composite engagement
---

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

DATA_PATH = "../data/processed"
OUT_PATH = "../data/processed"
MODEL_PATH = "../models"
os.makedirs(MODEL_PATH, exist_ok=True)

df = pd.read_csv(f"{DATA_PATH}/employee_attrition_processed.csv")
print(f"Loaded: {df.shape}")
print(f"Attrition distribution:")
print(df["Attrition"].value_counts())
df.head()

Loaded: (1470, 32)
Attrition distribution:
Attrition
No     1233
Yes     237
Name: count, dtype: int64


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Yes,11,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,No,23,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Yes,15,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Yes,11,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,No,12,3,4,1,6,3,3,2,2,2,2


---
## 1. Identify Leakage and Low-Value Columns
---

In [2]:
# Columns to drop — these leak the answer or are IDs/constants
drop_cols = [
    "EmployeeNumber",   # unique ID, not a feature
    "MonthlyRate",      # redundant pay period column
    "DailyRate",        # redundant pay period column
    "HourlyRate",       # redundant pay period column
]

print("Dropping leakage/non-predictive columns:")
for c in drop_cols:
    print(f"  {c}: {df[c].dtype} — {'unique ID' if c=='EmployeeNumber' else 'redundant pay period'}")

df_feat = df.drop(columns=drop_cols)
print(f"\nShape after dropping: {df_feat.shape}")

Dropping leakage/non-predictive columns:
  EmployeeNumber: int64 — unique ID
  MonthlyRate: int64 — redundant pay period
  DailyRate: int64 — redundant pay period
  HourlyRate: int64 — redundant pay period

Shape after dropping: (1470, 28)


---
## 2. Engineer New Features
---

In [3]:
# income_per_year: earning power relative to experience
df_feat["income_per_year"] = df_feat["MonthlyIncome"] * 12 / df_feat["TotalWorkingYears"].replace(0, 1)

# years_since_promotion_ratio: career stagnation signal
df_feat["years_since_promotion_ratio"] = df_feat["YearsSinceLastPromotion"] / df_feat["TotalWorkingYears"].replace(0, 1)

# satisfaction_score: composite of satisfaction-related ratings
df_feat["satisfaction_score"] = (
    df_feat["JobSatisfaction"]
    + df_feat["EnvironmentSatisfaction"]
    + df_feat["WorkLifeBalance"]
) / 3.0

print("Engineered features:")
for col in ["income_per_year", "years_since_promotion_ratio", "satisfaction_score"]:
    print(f"  {col}: mean={df_feat[col].mean():.2f}, std={df_feat[col].std():.2f}")
print(f"\nShape: {df_feat.shape}")

Engineered features:
  income_per_year: mean=8619.20, std=5823.86
  years_since_promotion_ratio: mean=0.20, std=0.26
  satisfaction_score: mean=2.74, std=0.57

Shape: (1470, 31)


---
## 3. Encode Categorical Variables
---

In [4]:
# Identify categorical columns
cat_cols = df_feat.select_dtypes(include=["object", "str"]).columns.tolist()
print(f"Categorical columns to encode: {cat_cols}")

# Ordinal encoding for binary Yes/No columns
binary_map = {"Yes": 1, "No": 0}
if "Attrition" in df_feat.columns:
    df_feat["Attrition"] = df_feat["Attrition"].map(binary_map)
    print(f"\nAttrition encoded: {df_feat['Attrition'].value_counts().to_dict()}")

if "OverTime" in df_feat.columns:
    df_feat["OverTime"] = df_feat["OverTime"].map(binary_map)
    print(f"OverTime encoded: {df_feat['OverTime'].value_counts().to_dict()}")

# One-hot encode remaining multi-class categoricals
multi_cat = [c for c in cat_cols if c not in ["Attrition", "OverTime"]]
print(f"\nOne-hot encoding: {multi_cat}")
df_feat = pd.get_dummies(df_feat, columns=multi_cat, drop_first=True, dtype=int)
print(f"Shape after encoding: {df_feat.shape}")

Categorical columns to encode: ['Attrition', 'BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']

Attrition encoded: {0: 1233, 1: 237}
OverTime encoded: {0: 1054, 1: 416}

One-hot encoding: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus']
Shape after encoding: (1470, 45)


---
## 4. Feature Summary
---

In [5]:
print(f"Final feature matrix: {df_feat.shape[0]} rows x {df_feat.shape[1]} columns")
print(f"\nAll columns:")
for i, c in enumerate(df_feat.columns):
    print(f"  {i+1}. {c} ({df_feat[c].dtype})")

# Save feature-engineered dataset
df_feat.to_csv(f"{OUT_PATH}/attrition_features.csv", index=False)
print(f"\nSaved: data/processed/attrition_features.csv ({df_feat.shape})")

Final feature matrix: 1470 rows x 45 columns

All columns:
  1. Age (int64)
  2. Attrition (int64)
  3. DistanceFromHome (int64)
  4. Education (int64)
  5. EnvironmentSatisfaction (int64)
  6. JobInvolvement (int64)
  7. JobLevel (int64)
  8. JobSatisfaction (int64)
  9. MonthlyIncome (int64)
  10. NumCompaniesWorked (int64)
  11. OverTime (int64)
  12. PercentSalaryHike (int64)
  13. PerformanceRating (int64)
  14. RelationshipSatisfaction (int64)
  15. StockOptionLevel (int64)
  16. TotalWorkingYears (int64)
  17. TrainingTimesLastYear (int64)
  18. WorkLifeBalance (int64)
  19. YearsAtCompany (int64)
  20. YearsInCurrentRole (int64)
  21. YearsSinceLastPromotion (int64)
  22. YearsWithCurrManager (int64)
  23. income_per_year (float64)
  24. years_since_promotion_ratio (float64)
  25. satisfaction_score (float64)
  26. BusinessTravel_Travel_Frequently (int64)
  27. BusinessTravel_Travel_Rarely (int64)
  28. Department_Research & Development (int64)
  29. Department_Sales (int64)
  

---
## 5. Feature Correlation with Target
---

In [6]:
# Correlation of all features with Attrition
corr = df_feat.corr(numeric_only=True)["Attrition"].drop("Attrition").sort_values()
print("Top negative correlations (less likely to leave):")
print(corr.head(5))
print()
print("Top positive correlations (more likely to leave):")
print(corr.tail(5))

Top negative correlations (less likely to leave):
TotalWorkingYears    -0.171063
JobLevel             -0.169105
YearsInCurrentRole   -0.160545
MonthlyIncome        -0.159840
satisfaction_score   -0.159721
Name: Attrition, dtype: float64

Top positive correlations (more likely to leave):
BusinessTravel_Travel_Frequently    0.115143
JobRole_Sales Representative        0.157234
income_per_year                     0.169047
MaritalStatus_Single                0.175419
OverTime                            0.246118
Name: Attrition, dtype: float64


---
## Feature Engineering Summary

| Action | Details |
| --- | --- |
| Dropped | EmployeeNumber, MonthlyRate, DailyRate, HourlyRate |
| Engineered | income_per_year, years_since_promotion_ratio, satisfaction_score |
| Encoded | OverTime (binary), Attrition (binary target), multi-class categoricals (one-hot) |
| Output | attrition_features.csv — ready for modeling |

**Next step:** Baseline Model (06_baseline_model.ipynb)
